```python
# Copyright (c) 2026 Louis Gevers
# SPDX-License-Identifier: BSD-3-Clause
```

# Legged Robots Project - Task 2: Flat Terrain Analysis

In this notebook you will analyse the data produced by `01_flat_terrain.ipynb`:
training logs and evaluation rollouts.
There are no simulations here, so it does not need a GPU and can be restarted at any time.

**Before you start:** check out the metrics functions in `src/lr_project/metrics.py`.
This is the only source file that you will have to edit.
We provided two example implementations, but it is up to you to implement the relevant metrics using rollout data (see 2.0c) for your analyses.
There are two kinds of functions, recognizable by their output shape (`N` number of robots, `T` number of timesteps):
- **per-step signals**, shape `(N, T)`: one value per robot and per time step, e.g. to plot a quantity over time.
- **per-robot metrics**, shape `(N,)`: one value per robot, summarizing its whole rollout.

Thanks to `%autoreload`, edits to `metrics.py` are picked up without restarting the kernel.

**Rules for your analysis**
- Every number and figure must come from a cell in this notebook. 
Never copy numbers by hand.
- The sections below list the **minimum** we expect. 
Add any analysis, figure or metric that helps you answer the questions.
- Label every figure: axes with units, a legend when there are several curves, and a title.
- Write your answers under the **[ANSWER]** cells in markdown.
Remember that short and precise beats long.

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import numpy as np
import matplotlib.pyplot as plt

from lr_project.rollout import Rollout
from lr_project import registry, analysis, metrics

## 2.0 Collected data

We provide you with an example of how to load three kinds of data you have generated:
the list of runs, the training logs of a run, and its evaluation rollouts.
Run it as is, then reuse the patterns in your analysis sections.

### 2.0a Selecting runs of interest

`registry.runs` lists runs filtered by experiment and variant (the names used in `train.train`), sorted by creation time. 
`runs[-1]` is the latest run (it's a python list).

In [ ]:
runs = registry.runs(
    experiment="task1_flat",
    variant="baseline",
)
registry.show(runs)

In [ ]:
run = runs[-1]
registry.show([run])

### 2.0b Training data

`analysis.load_training_logs` reads the curves logged during training and returns a dictionary `{curve name: (iterations, values)}`. 
`patterns` are regular expressions over the curve names, e.g. `"Episode_Reward/.*"` for all reward terms.
The curve names match the ones you see in TensorBoard.

In [ ]:
train_metrics = analysis.load_training_logs(
    run_dir=run.path,
    patterns=["Train/mean_reward", "Train/mean_episode_length", "Episode_Reward/.*"],
)

print("Loaded training metrics:")
for metric in train_metrics.keys():
    print(f"- {metric!r}")

Below we provide an example of plotting training logs, specifically the mean reward per learning iteration:

In [ ]:
iterations, mean_reward = train_metrics["Train/mean_reward"]
print(f"Loaded {len(iterations)} iterations of mean reward with last 3 elements: {mean_reward[-3:]}")

fig, ax = plt.subplots()

ax.plot(iterations, mean_reward)
ax.set_xlabel("Learning iteration")
ax.set_ylabel("Mean reward")

fig.tight_layout()
plt.show()

### 2.0c Evaluation data

`Rollout.load` reads an evaluation saved in notebook 01. 
A `Rollout` holds numpy arrays for N robots and T time steps (see `src/lr_project/rollout.py` for all fields), for example:

| Field | Shape | Content |
| --- | --- | --- |
| `cmd` | (N, 3) | command of each robot $(v_x, v_y, \omega_z)$ |
| `alive` | (N,) | `True` if the robot never fell |
| `base_lin_vel`, `base_ang_vel` | (N, T, 3) | base velocities in the body frame |
| `base_pos`, `base_quat` | (N, T, 3), (N, T, 4) | base position and orientation in the world frame |
| `joint_pos`, `joint_vel`, `joint_torque` | (N, T, 12) | joint states and torques |
| `foot_contact`, `foot_height` | (N, T, 4) | contact flags and heights of the feet (FL, FR, RL, RR) |
| `foot_grf` | (N, T, 4, 3) | ground reaction forces on the feet |

Robots that fell distort most metrics: compute them on `rollouts.select(rollouts.alive)` and report the survival rate separately.

Below we provide some examples of how to take statistics from rollout data and making a simple figure:

In [ ]:
rollouts = Rollout.load(run.eval_file("command_grid"))

n_robot = len(rollouts.alive)
survive_count = np.sum(rollouts.alive)
survival_rate = np.mean(rollouts.alive)

lin_vel_error = metrics.lin_vel_error(rollouts)

print(f"Survival rate is {survival_rate:.1%} ({np.sum(rollouts.alive)} out of {n_robot} evaluations alive)")
print(f"Average linear velocity error is {np.mean(lin_vel_error):.2f} (std={np.std(lin_vel_error):.2f})")

robot_id = 0
print(f"Robot {robot_id} command: {rollouts.cmd[robot_id]} (lin vel error: {lin_vel_error[robot_id]:.2f})")

fig, axes = plt.subplots(nrows=1, ncols=2)

time = rollouts.dt * np.arange(0, rollouts.base_pos.shape[1])
base_x = rollouts.base_pos[robot_id, :, 0]
base_y = rollouts.base_pos[robot_id, :, 1]
base_z = rollouts.base_pos[robot_id, :, 2]

axes[0].plot(base_x, base_y)
axes[0].set_aspect("equal")
axes[0].set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")

axes[1].plot(time, base_z)
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("z (m)")

fig.suptitle(f"Robot {robot_id} base trajectory")
fig.tight_layout()
plt.show()

## 2.1 Baseline performance

In this section you will characterize your baseline policy: how it learned, how well it performs, and discuss whether it could run on a real robot.

First load the training and rollout data:

In [ ]:
# TODO: Load your baseline training and rollout data here

### 2.1a Training performance

**Minimum:** mean reward and mean episode length curves, and the curves of the individual reward terms.

Questions: 
- When does the policy converge, and would more training help? 
- Which reward terms dominate, and which improve first? 
- Explain how you arrived at your baseline: which settings you tuned, what failures you observed along the way, and how you fixed them.

In [ ]:
# TODO: create your figures on training performance here

**[ANSWER]** *Write your answer to the questions here, 2 paragraphs max*

### 2.1b Evaluation performance

**Minimum:**
- velocity tracking: linear and yaw rate errors, overall and per command
- cost of transport
- base stability, with a metric of your own design
- foot ground reaction forces
- a foot contact plot (which foot is on the ground over time)

Questions: 
- Which commands are tracked well, and which are hardest? 
- Which gait did the policy learn, and how can you tell from the contact plot? 
- Is the motion efficient and smooth?

In [ ]:
# TODO: create your figures on evaluation performance here

**[ANSWER]** *Write your answer to the questions here, 2 paragraphs max*

### 2.1c Discussion: deployability

Given your previous observations, would your policy work on the real Go2? 
Discuss:
- **Observations:** can each observation be obtained on the real robot, and how noisy would it be?
- **Actions:** are the commanded motions realistic for the motors (look at torques, joint velocities and action smoothness)?
- **Evaluation on hardware:** which of your metrics could you measure on the real robot, and how?
- **Sim-to-real:** which reward terms or settings did you add to make the policy more transferable, and what is still missing?


**[ANSWER]** *Write your discussion here, 3 paragraphs max*

## 2.2 Controlled experiment

In this section you will describe and report the results of your controlled experiment in section 1.5 of notebook 01 like a short scientific study.

### 2.2a Questions and hypotheses

State your research question and your hypotheses:
- **Null hypothesis $H_0$:** the change has no effect on the metrics you chose.
- **Alternative hypothesis $H_1$:** the change affects the metrics, in a direction you predict, and why.

Name the metrics that will enable to reject the null hypothesis in favor of your alternative hypothesis, answering your research question.

**[ANSWER]** *Write your question, hypotheses, and metrics here.*

### 2.2b Methods

Describe exactly what differs between your baseline and your variant, what was held fixed (training budget, evaluation commands, seeds), how many runs each configuration has, and how you decide whether a difference is real (e.g. larger than the spread between seeds). 
Mention possible confounders.

**[ANSWER]** *Write your methods here. 2 paragraphs max.*

### 2.2c Experimental results

**Minimum:** at least 3 figures comparing baseline and variant, using training curves and/or evaluation metrics. 
Show both configurations in the same figure where possible, with the same axes.

In [ ]:
# TODO: Load your variant training and rollout data here

In [ ]:
# TODO: create your figures here

**[ANSWER]** *Describe what each figure shows here. 2 paragraphs max.*

### 2.2e Discussion

What did you expect, and what did you not expect? 
Do the results support $H_1$ and reject $H_0$, and how conclusive are they (number of runs, size of the difference compared to the variation between runs)? 
Explain the mechanism: how does your change lead to the effect you measured? 
What would you test next?

**[ANSWER]** *Describe your interpretations here. Linking your observations back to the course or the literature is encouraged! 3 paragraphs max.*